# 실습 11: 조정 다이얼 찾기
- 상황: 모델에는 사람이 정해줘야 하는 값이 있는데, 지금까지 손대지 않고 썼다
- 목표: 그 값을 손으로 돌려보고, 자동 탐색으로 찾아본다

## Step 0. 앞 실습까지 재현하기

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# 1. 정제본 불러오기 (day02 실습 결과물)
df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

# 2. 센서 열의 빈칸을 그 열의 중앙값으로 채우기
센서열 = [c for c in df.columns if c.startswith("sensor_")]
df[센서열] = df[센서열].fillna(df[센서열].median())

# 3. 판정을 숫자로 — 불량이면 1, 아니면 0
df["불량여부"] = (df["result"] == "불량").astype(int)

# 4. 입력은 센서 열만, 정답은 불량여부
X = df[센서열]
y = df["불량여부"]

# 5. 학습용과 시험용으로 나누기 (불량 비율을 양쪽에 맞춰서)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. 의사결정나무 — 손댄 설정은 두 개뿐이다
#    class_weight="balanced" : 드문 쪽(불량) 한 건을 더 무겁게 세라는 뜻
#    random_state=42         : 같은 나무가 나오도록 고정
#    나무는 열마다 따로 기준선을 긋기 때문에 표준화가 필요 없다
나무 = DecisionTreeClassifier(class_weight="balanced", random_state=42)
나무.fit(X_train, y_train)
나무예측 = 나무.predict(X_test)

# ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 나무예측).ravel()

print("학습용:", len(X_train), "건 (불량", int((y_train == 1).sum()), "건)")
print("시험용:", len(X_test), "건 (불량", int((y_test == 1).sum()), "건)")
print("정확도:", round((나무예측 == y_test).mean() * 100, 2), "%")
print("잡은 불량:", 잡은불량, "건")
print("놓친 불량:", 놓친불량, "건")
print("헛경보:", 헛경보, "건")

학습용: 1253 건 (불량 83 건)
시험용: 314 건 (불량 21 건)
정확도: 88.85 %
잡은 불량: 2 건
놓친 불량: 19 건
헛경보: 16 건


## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 사람이 정해주는 값

| 말 | 뜻 |
|---|---|
| 하이퍼파라미터 | 학습으로 정해지지 않고 사람이 미리 정해줘야 하는 값. 설명서에 이 이름으로 나온다 |
| max_depth | 나무가 몇 번까지 갈라질 수 있는지. 스무고개를 몇 번까지 할 것인가 |
| min_samples_leaf | 갈라진 끝자리에 최소 몇 건은 있어야 하는지 |
| 자동 탐색 | 후보를 적어주면 조합마다 다 돌려보고 점수를 재는 것 |
| 기준(scoring) | 자동 탐색이 1등을 뽑을 때 쓰는 자. 정하지 않으면 정확도로 뽑는다 |

## Step 2. 깊이를 손으로 바꿔보기

In [2]:
# 나무 모델과 채점 도구를 불러온다
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, recall_score, precision_score, f1_score

# 세 가지 깊이를 차례로 넣어본다. None 은 제한 없이 끝까지 간다는 뜻
for 깊이 in [3, 5, None]:
    # max_depth 자리만 바꾸고 나머지는 전부 같게 둔다
    나무 = DecisionTreeClassifier(random_state=42, class_weight="balanced", max_depth=깊이)
    나무.fit(X_train, y_train)
    예측 = 나무.predict(X_test)

    # ravel - 네 칸짜리 표를 한 줄로 펴서 이름을 하나씩 붙여 받는다
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 예측).ravel()

    print(f"깊이 {깊이}: 정확도 {round((예측 == y_test).mean() * 100, 2)}%",
          f"| 잡은 불량 {잡은불량} 놓친 불량 {놓친불량} 헛경보 {헛경보}",
          f"| 재현율 {round(recall_score(y_test, 예측), 3)}",
          f"F1 {round(f1_score(y_test, 예측), 3)}")

깊이 3: 정확도 48.09% | 잡은 불량 14 놓친 불량 7 헛경보 156 | 재현율 0.667 F1 0.147
깊이 5: 정확도 71.97% | 잡은 불량 10 놓친 불량 11 헛경보 77 | 재현율 0.476 F1 0.185
깊이 None: 정확도 88.85% | 잡은 불량 2 놓친 불량 19 헛경보 16 | 재현율 0.095 F1 0.103


### 문법 노트 - 다이얼 돌리기

| 쓴 것 | 하는 일 | 왜 여기 쓰나 |
|---|---|---|
| max_depth=3 | 세 번까지만 갈라지게 한다 | 얕게 두면 잘게 외우지 못하고 뭉뚱그려 판단한다 |
| max_depth=None | 제한을 두지 않는다 | 기본값. 답이 나올 때까지 끝까지 갈라진다 |
| random_state=42 | 갈라지는 과정의 무작위 요소를 고정한다 | 다시 돌려도 같은 결과가 나오게 |

## Step 3. 깊이별 결과

| 깊이 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 3 | [48.09]% | [14] | [156] | [0.667] | [0.147] |
| 5 | [71.97]% | [10] | [77] | [0.476] | [0.185] |
| 제한 없음 | [88.85]% | [2] | [16] | [0.095] | [0.103] |

## Step 4. 자동 탐색으로 찾기

In [3]:
# GridSearchCV - 후보를 적어주면 조합마다 다 돌려보고 점수를 재준다
from sklearn.model_selection import GridSearchCV

# 후보 - 딕셔너리로 적는다. 6가지 깊이 x 4가지 끝자리 최소건수 = 24조합
후보 = {
    "max_depth": [2, 3, 4, 5, 10, None],      # None = 제한 없음
    "min_samples_leaf": [1, 5, 10, 20],
}

탐색 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    후보,
    scoring="recall",   # 1등을 뽑는 자 - 재현율. 정하지 않으면 정확도로 뽑는다
    cv=5,               # 학습용을 다섯 조각으로 나눠 번갈아 채점한다 (교차검증)
)

# fit에 학습용만 넣는다. 시험용은 이 줄에 들어가지 않는다
탐색.fit(X_train, y_train)

print("[1. 1등으로 뽑힌 설정값]")
print(" ", 탐색.best_params_)

print("\n[2. 탐색 과정에서 나온 그 설정의 점수]")
print("  재현율(학습용 교차검증 평균):", round(탐색.best_score_, 3))

# 조합 24개를 점수순으로 놓고 위에서 몇 개만 본다
결과표 = pd.DataFrame(탐색.cv_results_)[
    ["param_max_depth", "param_min_samples_leaf", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
결과표.columns = ["max_depth", "min_samples_leaf", "재현율(평균)", "재현율(편차)", "순위"]
print("\n  탐색한 조합:", len(결과표), "개 / 상위 5개")
print(결과표.head(5).to_string(index=False))

# ---- 3. 1등 설정으로 시험용 채점 (여기서 처음으로 시험용을 쓴다) ----
# best_estimator_ - 1등 설정으로 학습용 전체를 다시 학습한 모델
최적나무 = 탐색.best_estimator_
최적예측 = 최적나무.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 최적예측).ravel()

print("\n[3. 1등 설정으로 시험용", len(y_test), "건을 채점한 결과]")
print("  정확도:", round((최적예측 == y_test).mean() * 100, 2), "%")
print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("  재현율:", round(recall_score(y_test, 최적예측), 3),
      "정밀도:", round(precision_score(y_test, 최적예측, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 최적예측), 3))

[1. 1등으로 뽑힌 설정값]
  {'max_depth': 3, 'min_samples_leaf': 20}

[2. 탐색 과정에서 나온 그 설정의 점수]
  재현율(학습용 교차검증 평균): 0.53

  탐색한 조합: 24 개 / 상위 5개
max_depth  min_samples_leaf  재현율(평균)  재현율(편차)  순위
        3                20 0.530147 0.177759   1
        3                10 0.518382 0.167382   2
        3                 1 0.518382 0.167382   2
        3                 5 0.518382 0.167382   2
        4                20 0.507353 0.176578   5

[3. 1등 설정으로 시험용 314 건을 채점한 결과]
  정확도: 48.09 %
  잡은 불량: 14 / 놓친 불량: 7 / 헛경보: 156
  재현율: 0.667 정밀도: 0.082 F1: 0.147


In [4]:
# 같은 후보, 같은 모델, 같은 cv — 1등을 뽑는 자(scoring)만 F1으로 바꾼다
# 앞의 탐색 결과(탐색, 최적예측)는 건드리지 않는다. 전부 새 이름으로 받는다

탐색_F1 = GridSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight="balanced"),
    후보,               # 위에서 만든 것 그대로 (max_depth 6가지 x min_samples_leaf 4가지)
    scoring="f1",       # 바꾼 곳은 이 한 줄뿐
    cv=5,
)

# 여기도 학습용만 넣는다
탐색_F1.fit(X_train, y_train)

print("[1. 이번에 1등으로 뽑힌 설정값]")
print(" ", 탐색_F1.best_params_)
print("  탐색 과정 점수 — F1(학습용 교차검증 평균):", round(탐색_F1.best_score_, 3))
print("  (앞 탐색의 1등:", 탐색.best_params_,
      "/ 재현율", round(탐색.best_score_, 3), ")")

# ---- 2. 그 설정으로 시험용 채점 ----
최적예측_F1 = 탐색_F1.best_estimator_.predict(X_test)

맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 최적예측_F1).ravel()

print("\n[2. F1 기준 1등 설정으로 시험용", len(y_test), "건을 채점한 결과]")
print("  정확도:", round((최적예측_F1 == y_test).mean() * 100, 2), "%")
print("  잡은 불량:", 잡은불량, "/ 놓친 불량:", 놓친불량, "/ 헛경보:", 헛경보)
print("  재현율:", round(recall_score(y_test, 최적예측_F1), 3),
      "정밀도:", round(precision_score(y_test, 최적예측_F1, zero_division=0), 3),
      "F1:", round(f1_score(y_test, 최적예측_F1), 3))

# ---- 3. 두 기준을 나란히 ----
def 한줄(기준, 탐색기, 답안지):
    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 답안지).ravel()
    return {
        "1등 뽑은 기준": 기준,
        "설정값": f"max_depth={탐색기.best_params_['max_depth']}, "
                  f"min_samples_leaf={탐색기.best_params_['min_samples_leaf']}",
        "탐색 점수": round(탐색기.best_score_, 3),
        "정확도(%)": round((답안지 == y_test).mean() * 100, 2),
        "잡은 불량": int(잡은불량),
        "놓친 불량": int(놓친불량),
        "헛경보": int(헛경보),
        "재현율": round(recall_score(y_test, 답안지), 3),
        "정밀도": round(precision_score(y_test, 답안지, zero_division=0), 3),
        "F1": round(f1_score(y_test, 답안지), 3),
    }

기준비교 = pd.DataFrame([
    한줄("재현율", 탐색, 최적예측),
    한줄("F1", 탐색_F1, 최적예측_F1),
]).set_index("1등 뽑은 기준")

print("\n[3. 두 기준 나란히]  시험용", len(y_test), "건 / 실제 불량", int((y_test == 1).sum()), "건")
print("(탐색 점수는 학습용 교차검증 값이라 서로 다른 자로 잰 값 — 두 줄을 직접 비교하면 안 된다)")
기준비교

[1. 이번에 1등으로 뽑힌 설정값]
  {'max_depth': 10, 'min_samples_leaf': 10}
  탐색 과정 점수 — F1(학습용 교차검증 평균): 0.192
  (앞 탐색의 1등: {'max_depth': 3, 'min_samples_leaf': 20} / 재현율 0.53 )

[2. F1 기준 1등 설정으로 시험용 314 건을 채점한 결과]
  정확도: 76.75 %
  잡은 불량: 6 / 놓친 불량: 15 / 헛경보: 58
  재현율: 0.286 정밀도: 0.094 F1: 0.141

[3. 두 기준 나란히]  시험용 314 건 / 실제 불량 21 건
(탐색 점수는 학습용 교차검증 값이라 서로 다른 자로 잰 값 — 두 줄을 직접 비교하면 안 된다)


,설정값,탐색 점수,정확도(%),잡은 불량,놓친 불량,헛경보,재현율,정밀도,F1
1등 뽑은 기준,,,,,,,,,
재현율,"max_depth=3, min_samples_leaf=20",0.530,48.09,14,7,156,0.667,0.082,0.147
F1,"max_depth=10, min_samples_leaf=10",0.192,76.75,6,15,58,0.286,0.094,0.141


## Step 5. 기준을 바꾸면 1등이 바뀐다

| 뽑은 기준 | 1등 설정 | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|---|
| 재현율 | [max_depth=3, min_samples_leaf=20] | [48.09]% | [14] | [156] | [0.667] | [0.147] |
| F1 | [max_depth=10, min_samples_leaf=10] | [76.75]% | [6] | [58] | [0.286] | [0.141] |

## Step 6. 내가 고른 설정

- 고른 기준 : [F1] - [놓친 불량도 줄이고 싶지만 헛경보 156건은 현장에서 감당이 안 될 것 같아서]
- 고른 설정 : [max_depth=10, min_samples_leaf=10]
- 이 설정의 시험용 성적 : [정확도 76.75% / 재현율 0.286 / 정밀도 0.094 / F1 0.141]

---
## 직접 해보기 (도전) - 다른 모델에도 다이얼이 있다

- 상황: 나무에만 다이얼이 있는 게 아니다
- 할 일: 로지스틱 회귀의 다이얼 하나를 네 값으로 돌려보고, 자동 탐색과 견줘본다
- 결과물: 네 줄짜리 표 1개 + 한 줄 메모

In [5]:
# 로지스틱 회귀에도 사람이 정해주는 값이 있다 (C, penalty, solver, class_weight, max_iter ...)
# 그중 C : 학습용에 얼마나 바짝 맞추게 둘지 조절하는 값 — 작을수록 느슨하게, 클수록 바짝 맞춘다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

행 = []
for C값 in [0.01, 0.1, 1, 10]:
    # C 자리만 바꾸고 나머지는 전부 같게 둔다
    회귀 = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", C=C값)
    )
    회귀.fit(X_train, y_train)          # 학습은 학습용으로만
    회귀예측 = 회귀.predict(X_test)     # 채점은 시험용으로

    맞힌양품, 헛경보, 놓친불량, 잡은불량 = confusion_matrix(y_test, 회귀예측).ravel()
    행.append({
        "C": C값,
        "정확도(%)": round((회귀예측 == y_test).mean() * 100, 2),
        "잡은 불량": int(잡은불량),
        "놓친 불량": int(놓친불량),
        "헛경보": int(헛경보),
        "재현율": round(recall_score(y_test, 회귀예측), 3),
        "정밀도": round(precision_score(y_test, 회귀예측, zero_division=0), 3),
        "F1": round(f1_score(y_test, 회귀예측), 3),
    })

C표 = pd.DataFrame(행).set_index("C")

print("시험용", len(y_test), "건 / 실제 불량", int((y_test == 1).sum()), "건")
print("C 말고는 아무것도 바꾸지 않았다 (표준화 + max_iter=1000 + class_weight='balanced' 고정)")
C표

시험용 314 건 / 실제 불량 21 건
C 말고는 아무것도 바꾸지 않았다 (표준화 + max_iter=1000 + class_weight='balanced' 고정)


,정확도(%),잡은 불량,놓친 불량,헛경보,재현율,정밀도,F1
C,,,,,,,
0.01,75.16,13,8,70,0.619,0.157,0.250
0.10,73.57,10,11,72,0.476,0.122,0.194
1.00,74.84,10,11,68,0.476,0.128,0.202
10.00,73.89,10,11,71,0.476,0.123,0.196


### 저울 쪽 모델의 다이얼

| C | 정확도 | 잡은 불량 | 헛경보 | 재현율 | F1 |
|---|---|---|---|---|---|
| 0.01 | [75.16]% | [13] | [70] | [0.619] | [0.250] |
| 0.1 | [73.57]% | [10] | [72] | [0.476] | [0.194] |
| 1 (기본값) | [74.84]% | [10] | [68] | [0.476] | [0.202] |
| 10 | [73.89]% | [10] | [71] | [0.476] | [0.196] |

- 알게 된 것 : [재현율로 보면 0.01이 제일 낫고, F1으로 보면 기본값 1이 제일 낫다. 여기서도 자에 따라 답이 갈린다]